# 🔬 Mini-projet — Jour  : Du vecteur au réseau de neurones
### Mathématiques appliquées à l'Intelligence Artificielle

2e année Bachelor Informatique — Année 2025-2026

---

> © 2025 Mohamed EL AFRIT
> Ce contenu est distribué sous licence [Creative Commons BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/deed.fr)
> [www.mohamedelafrit.com](https://www.mohamedelafrit.com)


## 🎯 Objectif

En **6 étapes**, vous allez appliquer toutes les notions des 4 jours sur le dataset fil rouge pour construire un système complet de prédiction du salaire.

| Étape | Thème | Jour |
|-------|-------|------|
| 1 | Exploration des données | Jour 1 |
| 2 | Régression linéaire from scratch | Jour 1 |
| 3 | Optimisation par descente de gradient | Jour 2 |
| 4 | Classification (perceptron) | Jour 3 |
| 5 | Réseau de neurones | Jour 4 |
| 6 | Synthèse et comparaison | Tous |


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# DATASET FIL ROUGE — 30 développeurs
# ============================================================
# Colonnes : experience, nb_langages, niveau_etudes,
#            taille_entreprise, remote, salaire
noms_colonnes = ["experience", "nb_langages", "niveau_etudes",
                 "taille_entreprise", "remote", "salaire"]

data = np.array([
    [ 0, 2, 3,  150,  80, 28.0],  # 0  junior
    [ 1, 1, 2,   50, 100, 26.5],  # 1  autodidacte
    [ 1, 3, 4,  800,  40, 35.0],  # 2  sortie ecole
    [ 2, 2, 3,  200,  60, 32.0],  # 3
    [ 2, 4, 4, 3000,  20, 38.0],  # 4
    [ 0, 1, 1,   30,  50, 25.0],  # 5  debutant
    [ 3, 3, 3,  100,  80, 34.5],  # 6
    [ 1, 5, 4,   15, 100, 42.0],  # 7  junior polyglotte
    [ 4, 3, 3,  500,  60, 38.5],  # 8
    [ 5, 4, 4, 1200,  40, 44.0],  # 9
    [ 5, 3, 3,  300,  80, 41.0],  # 10
    [ 6, 5, 4, 2000,  30, 48.0],  # 11
    [ 6, 4, 3,  150,  90, 43.5],  # 12
    [ 7, 5, 4,  800,  50, 50.0],  # 13
    [ 7, 3, 2,   80,  70, 39.0],  # 14 exp mais Bac+2
    [ 4, 6, 5,  400, 100, 47.0],  # 15
    [ 8, 4, 4, 1500,  40, 51.0],  # 16
    [ 5, 2, 3, 4000,  10, 40.0],  # 17
    [ 9, 5, 4,  600,  60, 52.0],  # 18
    [10, 6, 4, 2500,  30, 55.0],  # 19
    [10, 4, 3,  200,  80, 48.5],  # 20
    [11, 5, 5, 1000,  50, 58.0],  # 21
    [12, 7, 4, 3500,  20, 60.0],  # 22
    [ 9, 3, 3,   60,  90, 42.0],  # 23 senior sous-paye
    [13, 6, 4,  800,  70, 62.0],  # 24
    [11, 4, 4, 5000,  10, 54.0],  # 25
    [15, 8, 4, 1200,  50, 65.0],  # 26
    [17, 6, 5, 2000,  40, 68.0],  # 27
    [20, 7, 4,  500,  60, 72.0],  # 28
    [18, 5, 3,  100,  80, 58.0],  # 29
])

experience = data[:, 0]
nb_langages = data[:, 1]
salaire = data[:, 5]
salaire_eleve = (salaire > 45).astype(int)
n = len(data)

print(f"Dataset charge : {n} developpeurs, {data.shape[1]} colonnes")
print(f"Classe 0 (<=45k) : {np.sum(salaire_eleve==0)}, Classe 1 (>45k) : {np.sum(salaire_eleve==1)}")


---
# Étape 1 — Exploration des données (Jour 1)

> **Rappel** : La corrélation $r \in [-1, 1]$ mesure la liaison linéaire entre deux variables.


In [ ]:
# 1.1 Statistiques descriptives
print(f"{'Colonne':<20} {'Moy':>8} {'Std':>8} {'Min':>8} {'Max':>8}")
print("-" * 55)
for j, nom in enumerate(noms_colonnes):
    col = data[:, j] if j < 6 else salaire_eleve
    print(f"{nom:<20} {col.mean():>8.1f} {col.std():>8.1f} {col.min():>8.1f} {col.max():>8.1f}")


In [ ]:
# 1.2 Corrélations avec le salaire
print("=== Corrélations ===")
cors = []
for j in range(5):
    r = np.corrcoef(data[:,j], salaire)[0,1]
    cors.append(r)
    print(f"  {noms_colonnes[j]:<20} r = {r:+.3f}")

fig, axes = plt.subplots(1, 5, figsize=(18, 3.5))
for j in range(5):
    c = '#2ecc71' if abs(cors[j])>0.5 else '#e74c3c'
    axes[j].scatter(data[:,j], salaire, c=c, s=30, alpha=0.7)
    axes[j].set_xlabel(noms_colonnes[j], fontsize=8)
    axes[j].set_title(f"r={cors[j]:.2f}")
    axes[j].grid(True, alpha=0.3)
plt.suptitle("Corrélations features vs salaire", fontweight='bold')
plt.tight_layout(); plt.show()


❓ **Question** : Quelles sont les 2 features les plus corrélées au salaire ? Pourquoi la taille de l'entreprise est-elle faiblement corrélée ?


---
# Étape 2 — Régression linéaire from scratch (Jour 1)

> **Rappel** : $\hat{y} = w_1 x + w_0$, MSE $= \frac{1}{n}\sum(y_i - \hat{y}_i)^2$


In [ ]:
# 2.1 Régression experience → salaire (formule analytique)
x = experience; y = salaire
xm, ym = x.mean(), y.mean()
w1 = np.sum((x-xm)*(y-ym)) / np.sum((x-xm)**2)
w0 = ym - w1*xm
y_pred_reg = w1*x + w0
mse_reg = np.mean((y - y_pred_reg)**2)

print(f"Régression : ŷ = {w1:.4f}×exp + {w0:.4f}")
print(f"MSE = {mse_reg:.2f}, RMSE = {np.sqrt(mse_reg):.2f} k€")

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x, y, c='#3b82f6', s=50, label='Données')
xl = np.linspace(0, 21, 100)
ax.plot(xl, w1*xl+w0, 'r-', lw=2, label=f'ŷ={w1:.2f}x+{w0:.2f}')
ax.set_xlabel("Expérience"); ax.set_ylabel("Salaire (k€)")
ax.legend(); ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()


---
# Étape 3 — Optimisation par descente de gradient (Jour 2)

> **Rappel** : $w \leftarrow w - \alpha \frac{\partial J}{\partial w}$


In [ ]:
# 3.1 Gradient descent
x = experience; y = salaire
w1_gd, w0_gd = 0.0, 0.0; alpha = 0.001; hist_mse = []
for it in range(5000):
    err = y - (w1_gd*x + w0_gd)
    w1_gd -= alpha * (-2/n * np.sum(x*err))
    w0_gd -= alpha * (-2/n * np.sum(err))
    hist_mse.append(np.mean(err**2))

mse_gd = hist_mse[-1]
print(f"GD : ŷ = {w1_gd:.4f}×exp + {w0_gd:.4f}, MSE = {mse_gd:.2f}")

fig, (ax1,ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.plot(hist_mse, 'b-', lw=1.5); ax1.set_title("Convergence"); ax1.grid(True, alpha=0.3)
ax2.scatter(x, y, c='blue', s=40); xl=np.linspace(0,21,100)
ax2.plot(xl, w1_gd*xl+w0_gd, 'r-', lw=2, label=f'GD: {w1_gd:.2f}x+{w0_gd:.2f}')
ax2.legend(); ax2.grid(True, alpha=0.3); plt.tight_layout(); plt.show()


---
# Étape 4 — Classification : perceptron (Jour 3)

> **Rappel** : $\hat{y} = \text{signe}(\mathbf{w}^T\mathbf{x} + b)$


In [ ]:
# 4.1 Perceptron from scratch
X_2f = data[:, :2]; y_cls = salaire_eleve
w_perc = np.zeros(2); b_perc = 0.0
for _ in range(100):
    for i in range(n):
        z=w_perc@X_2f[i]+b_perc; yp=1 if z>0 else 0
        if yp!=y_cls[i]: e=y_cls[i]-yp; w_perc+=0.1*e*X_2f[i]; b_perc+=0.1*e

acc_perc = np.mean([(1 if (w_perc@X_2f[i]+b_perc)>0 else 0)==y_cls[i] for i in range(n)])
print(f"Perceptron accuracy : {acc_perc:.0%}")

fig, ax = plt.subplots(figsize=(8, 6))
for c,col,lab in [(0,'#e74c3c','≤45k'),(1,'#2ecc71','>45k')]:
    m=y_cls==c; ax.scatter(X_2f[m,0], X_2f[m,1], c=col, s=70, label=lab, edgecolors='white')
if abs(w_perc[1])>1e-6:
    x1=np.linspace(-1,21,100); ax.plot(x1, -(w_perc[0]/w_perc[1])*x1-b_perc/w_perc[1], 'k--', lw=2, label='Frontière')
ax.set_xlim(-1,21); ax.set_ylim(0,9); ax.legend(); ax.grid(True, alpha=0.3)
ax.set_title(f"Perceptron ({acc_perc:.0%})"); plt.tight_layout(); plt.show()


---
# Étape 5 — Réseau de neurones (Jour 4)

> **Rappel** : $\mathbf{h} = \sigma(\mathbf{W}^{(1)}\mathbf{x} + \mathbf{b}^{(1)})$, $\hat{y} = \sigma(\mathbf{w}^{(2)T}\mathbf{h} + b^{(2)})$


In [ ]:
# 5.1 MLP 2-4-1 from scratch
def sig(z): return 1/(1+np.exp(-np.clip(z,-500,500)))
def sig_d(a): return a*(1-a)

X_raw=data[:,:2]; y_r=salaire_eleve.reshape(-1,1).astype(float)
mu=X_raw.mean(axis=0); std=X_raw.std(axis=0); X_n=(X_raw-mu)/std

np.random.seed(42); W1=np.random.randn(2,4)*0.5; b1=np.zeros((1,4)); W2=np.random.randn(4,1)*0.5; b2=np.zeros((1,1))
hl=[]
for ep in range(2000):
    h=sig(X_n@W1+b1); out=sig(h@W2+b2); loss=np.mean((y_r-out)**2); hl.append(loss)
    do=(out-y_r)*sig_d(out); W2-=0.5*(h.T@do)/n; b2-=0.5*np.mean(do,axis=0,keepdims=True)
    dh=(do@W2.T)*sig_d(h); W1-=0.5*(X_n.T@dh)/n; b1-=0.5*np.mean(dh,axis=0,keepdims=True)

acc_mlp = np.mean((out>0.5)==y_r)
print(f"Réseau 2-4-1 accuracy : {acc_mlp:.0%}")

fig,(ax1,ax2)=plt.subplots(1,2,figsize=(14,5))
ax1.plot(hl,'b-'); ax1.set_title("Convergence"); ax1.grid(True,alpha=0.3)
xx,yy=np.meshgrid(np.linspace(X_n[:,0].min()-1,X_n[:,0].max()+1,100),np.linspace(X_n[:,1].min()-1,X_n[:,1].max()+1,100))
grid=np.c_[xx.ravel(),yy.ravel()]; zg=sig(sig(grid@W1+b1)@W2+b2).reshape(xx.shape)
ax2.contourf(xx,yy,zg,levels=20,cmap='RdYlGn',alpha=0.5); ax2.contour(xx,yy,zg,levels=[0.5],colors='black',linewidths=2)
for c,col in [(0,'#e74c3c'),(1,'#2ecc71')]:
    m=y_r.flatten()==c; ax2.scatter(X_n[m,0],X_n[m,1],c=col,s=60,edgecolors='white')
ax2.set_title(f"Surface de décision ({acc_mlp:.0%})"); plt.tight_layout(); plt.show()


---
# Étape 6 — Synthèse et conclusions


In [ ]:
print("=" * 60)
print("  BILAN DU MINI-PROJET")
print("=" * 60)
print(f"""
{'Étape':<30} {'Méthode':<22} {'Métrique':<15}
{'-'*67}
{'1. Exploration':<30} {'Corrélations':<22} {'r_exp=0.927':<15}
{'2. Régression (analytique)':<30} {f'ŷ={w1:.2f}x+{w0:.2f}':<22} {f'MSE={mse_reg:.2f}':<15}
{'3. Régression (GD)':<30} {f'ŷ={w1_gd:.2f}x+{w0_gd:.2f}':<22} {f'MSE={mse_gd:.2f}':<15}
{'4. Perceptron':<30} {'signe(w^Tx+b)':<22} {f'Acc={acc_perc:.0%}':<15}
{'5. Réseau 2-4-1':<30} {'MLP + backprop':<22} {f'Acc={acc_mlp:.0%}':<15}
""")

print("=== Ce que vous avez accompli ===")
print("J1 : REPRÉSENTER les données (vecteurs, matrices)")
print("J2 : OPTIMISER les paramètres (descente de gradient)")
print("J3 : CLASSIFIER avec un perceptron")
print("J4 : Construire un RÉSEAU DE NEURONES")
print("\n🎓 Vous avez les fondations mathématiques de l'IA !")
